# ChatGPT Images 2.6 정리: GIF 만드는 법, 2.5와 달라진 점, API 모델 Flare와 Sunburst 차이

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `OPENAI_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['OPENAI_API_KEY']:
    os.environ[name] = userdata.get(name)

### 설치

In [ ]:
%%bash
pip install -q pillow requests numpy
mkdir -p outputs

### Flare·Sunburst 로 프롬프트 GIF 생성 (OPENAI_API_KEY 환경변수 필요)

In [ ]:
%%bash
cat > gen.py <<'E'
import os,sys,time,json,base64,requests
model,tag=sys.argv[1],sys.argv[2]
prompt=("A cute round blue mascot blob with two eyes, flat vector style, plain white background, camera fixed. "
"It starts in a neutral standing pose, raises one arm to wave hello, then lowers it back to the exact starting pose so the loop is seamless.")
body=dict(model=model,prompt=prompt,output_format="gif",size="512x512",quality="medium",frame_count=9,frame_duration_ms=200)
t=time.time()
r=requests.post("https://api.openai.com/v1/images/generations",headers={"Authorization":"Bearer "+os.environ["OPENAI_API_KEY"]},json=body,timeout=600)
el=time.time()-t
print(r.status_code,round(el,1),"s")
j=r.json()
if r.status_code!=200: print(json.dumps(j)[:1500]); sys.exit(1)
print("usage",json.dumps(j.get("usage")))
open(f"outputs/{tag}.gif","wb").write(base64.b64decode(j["data"][0]["b64_json"]))
json.dump({"elapsed":el,"usage":j.get("usage")},open(f"outputs/{tag}.json","w"))
E
python gen.py gpt-image-2.6-flare flare_1
python gen.py gpt-image-2.6-sunburst sunburst_1
python - <<'E'
import json,os
from PIL import Image
for t in ['flare_1','sunburst_1']:
    im=Image.open(f'outputs/{t}.gif'); d=json.load(open(f'outputs/{t}.json')); u=d['usage']
    c=u['input_tokens']*5/1e6+u['output_tokens']*30/1e6
    print(t,im.format,im.size,'frames',im.n_frames,'dur',im.info.get('duration'),os.path.getsize(f'outputs/{t}.gif')//1024,'KB',round(d['elapsed'],1),'s',u['input_tokens'],u['output_tokens'],'$%.4f'%c)
E

### Sunburst 로 같은 프롬프트 GIF 생성 (Flare 와 비교)

In [ ]:
!python gen.py gpt-image-2.6-sunburst sunburst_1
!python gen.py gpt-image-2.6-flare flare_2
!python gen.py gpt-image-2.6-sunburst sunburst_2

### 네 결과의 프레임 수·용량·시간·토큰 비교 + 프레임 시트 저장

In [ ]:
%%bash
cat > an.py <<'E'
import json,os
from PIL import Image
for t in ["flare_1","flare_2","sunburst_1","sunburst_2"]:
    p=f"outputs/{t}.gif"; im=Image.open(p); d=json.load(open(f"outputs/{t}.json")); u=d["usage"]
    c=u["input_tokens"]*5/1e6+u["output_tokens"]*30/1e6
    print(t,im.format,im.size,"frames",im.n_frames,"dur",im.info.get("duration"),"loop",im.info.get("loop"),os.path.getsize(p)//1024,"KB",round(d["elapsed"],1),"s",u["input_tokens"],u["output_tokens"],"$%.4f"%c)
    n=im.n_frames; sheet=Image.new("RGB",(256*n,256),"white")
    for i in range(n):
        im.seek(i); sheet.paste(im.convert("RGB").resize((256,256)),(256*i,0))
    sheet.save(f"outputs/sheet_{t}.png")
E
python an.py

### 직접 만든 정지 이미지(마스코트 + HELLO 글자) 생성

In [ ]:
from PIL import Image,ImageDraw,ImageFont
f=ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",60)
im=Image.new("RGB",(512,512),"white");d=ImageDraw.Draw(im)
d.ellipse((136,150,376,390),fill=(60,120,230));d.ellipse((190,230,225,265),fill="white");d.ellipse((290,230,325,265),fill="white")
d.ellipse((200,240,215,255),fill="black");d.ellipse((300,240,315,255),fill="black");d.text((256,60),"HELLO",font=f,fill="black",anchor="mt")
im.save("outputs/input_mascot.png")
im

### 정지 이미지를 입력으로 GIF 를 만드는 edit.py 작성

In [ ]:
%%writefile edit.py
import os,sys,time,json,base64,requests
model,tag=sys.argv[1],sys.argv[2]
t=time.time()
r=requests.post("https://api.openai.com/v1/images/edits",headers={"Authorization":"Bearer "+os.environ["OPENAI_API_KEY"]},
 files={"image":open("outputs/input_mascot.png","rb")},
 data={"model":model,"prompt":"Animate: the mascot waves one arm, then returns to the start pose. Keep the HELLO text and mascot shape unchanged. Camera fixed, seamless loop.","output_format":"gif","frame_count":"9","frame_duration_ms":"200","size":"512x512","quality":"medium"})
el=time.time()-t;j=r.json();print(r.status_code,round(el,1),json.dumps(j.get("usage")))
open(f"outputs/{tag}.gif","wb").write(base64.b64decode(j["data"][0]["b64_json"]))
json.dump({"elapsed":el,"usage":j.get("usage")},open(f"outputs/{tag}.json","w"))

### Flare·Sunburst 로 정지 이미지 → GIF 생성

In [ ]:
!python edit.py gpt-image-2.6-flare edit_flare
!python edit.py gpt-image-2.6-sunburst edit_sunburst

### 원본 대비 프레임별 평균 픽셀 차이 계산, 프레임 시트 저장

In [ ]:
import os
from PIL import Image
import numpy as np
for t in ["edit_flare","edit_sunburst"]:
    im=Image.open(f"outputs/{t}.gif");print(t,im.format,im.size,im.n_frames,im.info.get("duration"),os.path.getsize(f"outputs/{t}.gif")//1024,"KB")
    n=im.n_frames;sheet=Image.new("RGB",(256*n,256),"white");ref=Image.open("outputs/input_mascot.png").convert("RGB")
    ds=[]
    for i in range(n):
        im.seek(i);f=im.convert("RGBA");bg=Image.new("RGBA",f.size,"white");bg.alpha_composite(f);f=bg.convert("RGB")
        ds.append(round(float(np.abs(np.asarray(f,float)-np.asarray(ref,float)).mean()),1));sheet.paste(f.resize((256,256)),(256*i,0))
    sheet.save(f"outputs/sheet_{t}.png");print(ds)
Image.open("outputs/sheet_edit_flare.png")

### 응답 usage 와 단가($5/$30 per 1M 토큰)로 GIF 한 개 비용 계산

In [ ]:
import json
for t in ["flare_1","sunburst_1","edit_flare","edit_sunburst"]:
    u=json.load(open(f"outputs/{t}.json"))["usage"]
    c=u["input_tokens"]*5/1e6+u["output_tokens"]*30/1e6
    print(t,"입력",u["input_tokens"],"출력",u["output_tokens"],"$%.4f"%c)